# Service MDP — Iteration de la valeur — Kernel Gateway

Exposition REST de l'**iteration de la valeur** (backups de Bellman + politique gloutonne) du carnet phare [RL-05 — MDP, programmation dynamique et Q-learning](../RL-05-MDP-Programmation-Dynamique-Q-Learning-Python.ipynb) — programmation dynamique sur MDP, chapitre 17 d'AIMA.

Feuille de service de l'Epic AIMA-Gateway #19571. Le carnet pedagogique n'est pas modifie : ce dossier `gateway/` ajoute un carnet de service a cote, sur le meme motif que les pilotes `GameTheory/gateway/`, `Probas/Infer/gateway/`, `Z3-API/gateway/` et `NLP/gateway/`.

**Contrat** : `POST /rl/mdp-value-iteration` — corps = MDP (etats, actions, transitions `T`, recompenses `R`, `gamma`), reponse = politique optimale + fonction de valeur. Le detail du contrat, la variante du gridworld 4x3 de demonstration et le verdict SOTA sont dans le [README](README.md) de ce dossier.

In [1]:
# Variante du gridworld 4x3 d'AIMA : deux obstacles et deux terminaux.
import importlib.util
from pathlib import Path

_core_path = Path("mdp_value_iteration_core.py")
_spec = importlib.util.spec_from_file_location("mdp_value_iteration_core", _core_path)
core = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(core)
run = core.run


def build_gridworld():
    """Récompense à l'entrée du terminal, puis transition absorbante sans récompense."""
    cols, rows = 4, 3
    obstacles = {(1, 1), (1, 3)}
    terminal_reward = {(0, 3): 1.0, (2, 0): -1.0}
    step_cost = -0.04
    cells = [(r, c) for r in range(rows) for c in range(cols) if (r, c) not in obstacles]
    states = [f"{r},{c}" for r, c in cells]
    actions = ["N", "S", "E", "W"]
    delta = {"N": (-1, 0), "S": (1, 0), "E": (0, 1), "W": (0, -1)}

    def in_bounds(r, c):
        return 0 <= r < rows and 0 <= c < cols

    T, R = {}, {}
    for r, c in cells:
        state = f"{r},{c}"
        T[state], R[state] = {}, {}
        for a in actions:
            if (r, c) in terminal_reward:
                T[state][a] = {state: 1.0}
                R[state][a] = 0.0
                continue
            dr, dc = delta[a]
            perps = [(-1, 0), (1, 0)] if a in ("N", "S") else [(0, -1), (0, 1)]
            outcomes = {}
            for (dy, dx), probability in [((dr, dc), 0.8), (perps[0], 0.1), (perps[1], 0.1)]:
                rr, cc = r + dy, c + dx
                if not in_bounds(rr, cc) or (rr, cc) in obstacles:
                    rr, cc = r, c
                destination = f"{rr},{cc}"
                outcomes[destination] = outcomes.get(destination, 0.0) + probability
            T[state][a] = outcomes
            R[state][a] = sum(
                probability * terminal_reward.get(tuple(map(int, destination.split(","))), step_cost)
                for destination, probability in outcomes.items()
            )
    return {"states": states, "actions": actions, "T": T, "R": R, "gamma": 0.9}


mdp = build_gridworld()
print("Coeur charge :", _core_path.name)
print("Etats       :", len(mdp["states"]), "(gridworld 4x3, 2 obstacles)")
print("Actions     :", mdp["actions"])

Coeur charge : mdp_value_iteration_core.py
Etats       : 10 (gridworld 4x3, 2 obstacles)
Actions     : ['N', 'S', 'E', 'W']


In [2]:
# POST /rl/mdp-value-iteration
#
# Point d'entree du service. En mode gateway, kernel_gateway route les POST sur
# /rl/mdp-value-iteration vers `http_server` ; le corps JSON est passe a `run()` du coeur et la
# reponse est serialisee. Le carnet s'execute aussi hors gateway
# (papermill/nbconvert) : `kernel_gateway` n'est alors pas importe.
import json

try:  # mode gateway : le type HTTPRequest existe
    from kernel_gateway import HTTPRequest  # noqa: F401
    _GATEWAY = True
except ImportError:  # mode hors gateway (papermill/nbconvert)
    _GATEWAY = False


def http_server(request, kernel_session):
    """Route `POST /rl/mdp-value-iteration` : `run()` du coeur, reponse JSON au contrat."""
    try:
        body = json.loads(request.body)
    except Exception as exc:
        payload = {"status": "invalid", "error": f"corps JSON invalide: {exc}"}
        return 200, json.dumps(payload, ensure_ascii=False).encode("utf-8")
    return 200, json.dumps(run(body), ensure_ascii=False).encode("utf-8")


print("Mode gateway :", "oui" if _GATEWAY else "non (demo hors gateway)")

Mode gateway :

non (demo hors gateway)

In [3]:
# Demonstration : iteration de la valeur sur le gridworld 4x3 (sortie reelle).
import json

response = run(mdp)
print("Iterations (backups de Bellman) :", response["iterations"])
print("Converge                        :", response["converged"])
print("Politique optimale (extrait)    :")
for s in ("0,0", "0,3", "2,0", "2,3"):
    print(f"  {s} -> {response['policy'][s]}")
print("Valeurs optimales (extrait)     :")
for s in ("0,0", "0,3", "2,0", "2,3"):
    print(f"  V({s}) = {response['value'][s]:.4f}")

Iterations (backups de Bellman) :

19

Converge                        :

True

Politique optimale (extrait)    :

  0,0 -> E

... (4 outputs left out)

  V(0,0) = 0.6726

  V(0,3) = 0.0000

  V(2,0) = 0.0000

  V(2,3) = 0.5547

In [4]:
# Demonstration : entree hors contrat -> `status: invalid`, aucune exception.
import json

bad = json.loads(json.dumps(mdp))  # copie profonde
bad["T"]["0,0"]["N"]["0,0"] = 0.5  # distribution de transition non normalisee
print(json.dumps(run(bad), ensure_ascii=False, indent=2))

{
  "status": "invalid",
  "error": "T['0,0']['N']: somme = 0.600000, distribution normalisee (somme ~ 1) attendue"
}

## Lecture du résultat

L'itération de la valeur applique des **backups de Bellman** jusqu'à convergence, puis construit une politique gloutonne. Dans cette variante du gridworld 4x3 d'AIMA, la récompense `+1` ou `-1` est reçue à l'entrée d'un terminal. Ensuite, l'état reste sur lui-même avec une récompense nulle : sa valeur est donc nulle et l'action affichée y est arbitraire.

**Verdict SOTA : SOTA-OK.** Cette copie pédagogique autonome expose l'algorithme enseigné dans RL-05 ; la géométrie et la convention de récompense de l'exemple constituent une variante de la grille.